# Week 57, Day 6: Forge: Week 57 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w057/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
from math import comb
import numpy as np
print("1.", comb(10 + 2, 2))
print("2.", round(np.exp(-0.5 * 2 ** 2), 4))
print("3.", 12 / 200)

### Exercise: Part A: nested cross-validation of three decoders

The SVM solver, kernels and data are provided. In each outer fold, choose gamma for the Gaussian SVM by three-fold cross-validation on the outer training trials only, refit, and score on the outer test fold. Do the same for the linear and random-feature decoders (no inner choice needed). Store mean outer accuracies.

In [ ]:
import numpy as np
r = np.random.default_rng(576)

def svm_dual(K, Y, C=1.0, sweeps=40):
    n = len(Y); a = np.zeros(n); f = np.zeros(n)
    for s in range(sweeps):
        for i in range(n):
            new = min(max(a[i] - (Y[i] * f[i] - 1) / K[i, i], 0.0), C)
            if new != a[i]:
                f += (new - a[i]) * Y[i] * K[:, i]; a[i] = new
    return a

def gauss_k(A, B, g):
    return np.exp(-g * np.maximum((A ** 2).sum(1)[:, None] - 2 * A @ B.T + (B ** 2).sum(1)[None, :], 0)) + 1

n, D = 300, 8
X = r.normal(size=(n, D))
q = (X[:, :2] ** 2 / np.array([1.0, 2.0])).sum(1)
Y = np.where(q < np.median(q), 1.0, -1.0)                 # inside an ellipse in 2 of 8 dimensions
W_rf = r.normal(size=(300, D)) * np.sqrt(0.2); b_rf = r.uniform(0, 2 * np.pi, 300)
rf = lambda A: np.sqrt(2 / 300) * np.cos(A @ W_rf.T + b_rf)

def score(K_tr, Y_tr, K_te, Y_te):
    a = svm_dual(K_tr, Y_tr)
    return np.mean(np.sign(K_te @ (a * Y_tr)) == Y_te)

gammas = [0.01, 0.05, 0.2, 1.0]
outer = {"linear": [], "gaussian": [], "random features": []}
for te in np.array_split(np.arange(n), 5):
    tr = np.setdiff1d(np.arange(n), te)
    outer["linear"].append(score(X[tr] @ X[tr].T + 1, Y[tr], X[te] @ X[tr].T + 1, Y[te]))
    F_tr, F_te = rf(X[tr]), rf(X[te])
    outer["random features"].append(score(F_tr @ F_tr.T + 1, Y[tr], F_te @ F_tr.T + 1, Y[te]))
    inner = []
    for g in gammas:
        acc_g = []
        for ite in np.array_split(tr, 3):
            itr = np.setdiff1d(tr, ite)
            acc_g.append(...)
        inner.append(np.mean(acc_g))
    g_best = gammas[int(np.argmax(inner))]
    outer["gaussian"].append(...)
outer = {k: float(np.mean(v)) for k, v in outer.items()}
for k, v in outer.items():
    print(f"{k:16s} outer cross-validated accuracy {v:.3f}")
print(f"(chance is {max(np.mean(Y > 0), np.mean(Y < 0)):.3f})")

In [ ]:
import numpy as np
def test_keys():
    "outer has a mean accuracy for each decoder"
    assert set(outer) == {"linear", "gaussian", "random features"}
def test_order():
    "both nonlinear decoders beat the linear one by at least 10 points"
    assert outer["gaussian"] > outer["linear"] + 0.1 and outer["random features"] > outer["linear"] + 0.1

test_keys()
test_order()
print("All checks passed")

### Exercise: Part B: the support-vector bound

For each C, fit the Gaussian SVM (gamma = 2) to all 80 examples and count support vectors; then compute the leave-one-out error by refitting without each example in turn.

In [ ]:
def moons(n, g):
    y = np.where(g.random(n) < 0.5, -1.0, 1.0); a = np.pi * g.random(n)
    x = np.c_[np.where(y > 0, np.cos(a), 1 - np.cos(a)), np.where(y > 0, np.sin(a), 0.5 - np.sin(a))]
    return x + 0.25 * g.normal(size=(n, 2)), y

Xm, Ym = moons(80, r)
Km = gauss_k(Xm, Xm, 2.0)
results = {}
for C in [0.3, 1.0, 10.0]:
    a_all = svm_dual(Km, Ym, C, sweeps=80)
    frac_sv = np.mean(a_all > 1e-8)
    errors = 0
    for i in range(80):
        keep = np.arange(80) != i
        a = svm_dual(Km[np.ix_(keep, keep)], Ym[keep], C, sweeps=80)
        errors += ...
    results[C] = (errors / 80, frac_sv)
    print(f"C = {C:5.1f}: leave-one-out error {errors / 80:.3f}; support-vector fraction {frac_sv:.3f}")

In [ ]:
import numpy as np
def test_bound():
    "for every C, the leave-one-out error does not exceed the fraction of support vectors"
    assert all(loo <= sv + 1e-9 for loo, sv in results.values())

test_bound()
print("All checks passed")